# MyDigitalTwin — 02_topology / 01 Enrichissement

Enrichit les sources sans texte avant le pipeline topology.

**Sources enrichies :**
- `tiktok_likes` + `tiktok_saves` → description vidéo via `og:description` → oEmbed fallback
- `instagram_saved` → biographie du compte via `instaloader`

**Outputs (cache disque, one-shot) :**
- `data/warehouse/tiktok_descriptions.json`  — {video_id: description | null}
- `data/warehouse/instagram_bios.json`        — {username: "username bio_text"}

> Ce notebook est **one-shot** : une fois les caches créés, pas besoin de le relancer
> sauf si tu as de nouvelles données (nouvel export GDPR).

### Dépendances
```bash
pip install aiohttp beautifulsoup4 instaloader nest_asyncio
```

In [1]:
import sys, os, json, time, re
import nest_asyncio
nest_asyncio.apply()  # permet asyncio.run() dans Jupyter

# ── Config paths ──────────────────────────────────────────────────────────────
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import WAREHOUSE

TIKTOK_CACHE_PATH    = os.path.join(WAREHOUSE, "tiktok_descriptions.json")
INSTAGRAM_CACHE_PATH = os.path.join(WAREHOUSE, "instagram_bios.json")

def load_cache(path: str) -> dict:
    if os.path.isfile(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(data: dict, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  Cache sauvegardé : {path}  ({len(data)} entrées)")

print("\u2713 Config chargée")
print(f"  WAREHOUSE = {WAREHOUSE}")
print(f"  TikTok cache  : {TIKTOK_CACHE_PATH}")
print(f"  Instagram cache: {INSTAGRAM_CACHE_PATH}")

✓ Config chargée
  WAREHOUSE = /opt/spark/data/warehouse
  TikTok cache  : /opt/spark/data/warehouse/tiktok_descriptions.json
  Instagram cache: /opt/spark/data/warehouse/instagram_bios.json


---
## 1. TikTok — Descriptions vidéo

Stratégie par ordre de priorité :
1. `og:description` depuis la page de partage TikTok (HTTP GET simple)
2. `tiktok.com/oembed?url=...` → champ `title` (endpoint semi-officiel, pas de clé)
3. `None` → le point sera traité en fallback comportemental dans `02_graph.ipynb`

In [2]:
import pandas as pd

# Charger les URLs depuis le warehouse
dfs = []
for table in ["tiktok_likes", "tiktok_saves"]:
    p = os.path.join(WAREHOUSE, table)
    if os.path.exists(p):
        df = pd.read_parquet(p)
        dfs.append(df[["video_id", "url"]].drop_duplicates("video_id"))
        print(f"\u2713 {table:<20} {len(df):>6,} items")
    else:
        print(f"\u26a0\ufe0f  {table} absent du warehouse \u2014 skip")

if not dfs:
    raise RuntimeError("Aucune table TikTok trouv\u00e9e \u2014 relancer tiktok.ipynb d'abord")

df_tiktok = pd.concat(dfs, ignore_index=True).drop_duplicates("video_id")
print(f"\nURLs uniques total : {len(df_tiktok):,}")

tiktok_cache = load_cache(TIKTOK_CACHE_PATH)
missing      = df_tiktok[~df_tiktok["video_id"].isin(tiktok_cache)].copy()
print(f"D\u00e9j\u00e0 en cache    : {len(tiktok_cache):,}")
print(f"\u00c0 fetcher        : {len(missing):,}")

✓ tiktok_likes         12,000 items
✓ tiktok_saves             72 items

URLs uniques total : 6,065
Déjà en cache    : 6,065
À fetcher        : 0


In [3]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
}
CONCURRENCY = 30
TIMEOUT     = 8


async def _fetch_og(session, video_id, url, sem):
    """Essai 1 : og:description depuis la page de partage."""
    async with sem:
        try:
            async with session.get(url, headers=HEADERS,
                                   timeout=aiohttp.ClientTimeout(total=TIMEOUT)) as r:
                if r.status != 200:
                    return video_id, None
                html = await r.text(errors="replace")
            soup = BeautifulSoup(html, "html.parser")
            tag  = (soup.find("meta", property="og:description")
                    or soup.find("meta", attrs={"name": "description"}))
            if tag and tag.get("content", "").strip():
                return video_id, tag["content"].strip()
        except Exception:
            pass
    return video_id, None


async def _fetch_oembed(session, video_id, url, sem):
    """Essai 2 : TikTok oEmbed → champ 'title'."""
    oembed = f"https://www.tiktok.com/oembed?url={url}"
    async with sem:
        try:
            async with session.get(oembed, headers=HEADERS,
                                   timeout=aiohttp.ClientTimeout(total=TIMEOUT)) as r:
                if r.status != 200:
                    return video_id, None
                data  = await r.json(content_type=None)
                title = data.get("title", "").strip()
                if title:
                    return video_id, title
        except Exception:
            pass
    return video_id, None


async def _enrich_all(rows: pd.DataFrame) -> dict:
    results = {}
    sem     = asyncio.Semaphore(CONCURRENCY)
    async with aiohttp.ClientSession() as session:
        # Étape 1 : og:description
        og_tasks = [
            _fetch_og(session, row["video_id"], row["url"], sem)
            for _, row in rows.iterrows()
        ]
        print(f"[og:description] {len(og_tasks):,} URLs...")
        og_res = await asyncio.gather(*og_tasks)

        failed = []
        for (vid, desc), (_, row) in zip(og_res, rows.iterrows()):
            if desc:
                results[vid] = desc
            else:
                failed.append((vid, row["url"]))
        print(f"  \u2713 ok={len(results):,}  \u00e9checs={len(failed):,} \u2192 oEmbed")

        # Étape 2 : oEmbed sur les échecs
        if failed:
            oe_tasks = [
                _fetch_oembed(session, vid, url, sem)
                for vid, url in failed
            ]
            oe_res = await asyncio.gather(*oe_tasks)
            ok_oe  = sum(1 for _, v in oe_res if v)
            for vid, title in oe_res:
                results[vid] = title  # None si échec d\u00e9finitif
            print(f"  \u2713 oEmbed ok={ok_oe:,}  \u00e9checs d\u00e9finitifs={len(failed)-ok_oe:,}")

    return results


# ── Lancement ─────────────────────────────────────────────────────────────────
if len(missing) > 0:
    t0   = time.time()
    news = asyncio.run(_enrich_all(missing))
    tiktok_cache.update(news)
    save_cache(tiktok_cache, TIKTOK_CACHE_PATH)

    ok = sum(1 for v in news.values() if v)
    print(f"\n\u2705 Termin\u00e9 en {time.time()-t0:.0f}s")
    print(f"   Descriptions r\u00e9cup\u00e9r\u00e9es : {ok:,} / {len(news):,} "
          f"({100*ok/max(1,len(news)):.0f}%)")
    print(f"   Fallback comportemental  : {len(news)-ok:,}")
else:
    print("\u2713 Tout d\u00e9j\u00e0 en cache \u2014 rien \u00e0 fetcher")

✓ Tout déjà en cache — rien à fetcher


In [4]:
# Stats cache TikTok
filled = {k: v for k, v in tiktok_cache.items() if v}
empty  = {k: v for k, v in tiktok_cache.items() if not v}

print(f"Cache TikTok total    : {len(tiktok_cache):,}")
print(f"  \u2192 avec description  : {len(filled):,} ({100*len(filled)/max(1,len(tiktok_cache)):.0f}%)")
print(f"  \u2192 sans description  : {len(empty):,}  (fallback comportemental)")
print()
print("Exemples de descriptions r\u00e9cup\u00e9r\u00e9es :")
for k, v in list(filled.items())[:5]:
    print(f"  [{k[:10]}...] {str(v)[:90]}")

Cache TikTok total    : 6,065
  → avec description  : 0 (0%)
  → sans description  : 6,065  (fallback comportemental)

Exemples de descriptions récupérées :


---
## 2. Instagram — Biographies des comptes

Instagram bloque toutes les requêtes graphql sans authentification (403) depuis 2024.
`instaloader` **nécessite une session** pour fonctionner.

### Setup one-shot (une seule fois)
```bash
# Dans un terminal (hors Jupyter) :
instaloader --login=TON_USERNAME_INSTAGRAM
# → saisir le mot de passe
# → session sauvegardée dans ~/.config/instaloader/session-TON_USERNAME
```
Le cookie est persistant (pas besoin de relogger). Aucun mot de passe dans le notebook.

### Ce que fait cette section
- Charge la session depuis le fichier cookie
- Fetche la bio des 17 comptes `instagram_saved`
- Rate limit avec login : ~1000 profils/heure → délai réduit à 5s
- Cache disque : `warehouse/instagram_bios.json` — jamais refetché si déjà présent
- Comptes privés / supprimés → username seul (embedding faible mais présent)

In [5]:
import pandas as pd

try:
    import instaloader
    IL_AVAILABLE = True
    print("✓ instaloader disponible")
except ImportError:
    IL_AVAILABLE = False
    print("⚠️  instaloader non installé  →  pip install instaloader")

# ── Username Instagram depuis config.yaml ─────────────────────────────────────
from config import INSTAGRAM_USERNAME
IG_USERNAME = INSTAGRAM_USERNAME
print(f"IG_USERNAME = {IG_USERNAME!r}")

# ── Charger les comptes depuis instagram_saved ────────────────────────────────
ig_path = os.path.join(WAREHOUSE, "instagram_saved")
if os.path.exists(ig_path):
    df_ig = pd.read_parquet(ig_path)
    accounts = (
        df_ig["account"].dropna()
        .str.lower().str.strip()
        .unique().tolist()
    )
    print(f"Comptes uniques instagram_saved : {len(accounts):,}")
else:
    accounts = []
    print("⚠️  instagram_saved absent du warehouse")

ig_cache  = load_cache(INSTAGRAM_CACHE_PATH)
to_fetch  = [a for a in accounts if a not in ig_cache]
print(f"Déjà en cache : {len(ig_cache):,}")
print(f"À fetcher     : {len(to_fetch):,}")

# ── Vérifier la session ────────────────────────────────────────────────────────
if IL_AVAILABLE and to_fetch and IG_USERNAME:
    session_path = os.path.expanduser(
        f"~/.config/instaloader/session-{IG_USERNAME}"
    )
    if not os.path.exists(session_path):
        print(f"\n⚠️  Pas de session sauvegardée pour @{IG_USERNAME}")
        print(f"   Lance une fois dans un terminal :")
        print(f"   instaloader --login={IG_USERNAME}")
        print(f"   (mot de passe → session sauvegardée dans {session_path})")
        IL_AVAILABLE = False
    else:
        print(f"✓ Session trouvée : {session_path}")
elif IL_AVAILABLE and to_fetch and not IG_USERNAME:
    print("\n⚠️  instagram_username manquant dans config.yaml")

✓ instaloader disponible
IG_USERNAME = 'arnvudl'
Comptes uniques instagram_saved : 17
Déjà en cache : 0
À fetcher     : 17
✓ Session trouvée : /root/.config/instaloader/session-arnvudl


In [6]:
DELAY = 5   # secondes entre requêtes (avec login : rate limit ~1000/h)

if IL_AVAILABLE and to_fetch and IG_USERNAME:
    L = instaloader.Instaloader(
        download_pictures=False,
        download_videos=False,
        download_video_thumbnails=False,
        download_geotags=False,
        download_comments=False,
        save_metadata=False,
        quiet=True,
    )

    # ── Charger la session (cookie) ───────────────────────────────────────────
    session_path = os.path.expanduser(
        f"~/.config/instaloader/session-{IG_USERNAME}"
    )
    L.load_session_from_file(IG_USERNAME, session_path)
    print(f"✓ Connecté en tant que @{IG_USERNAME}")
    print(f"Fetching {len(to_fetch)} biographies...\n")

    for i, username in enumerate(to_fetch):
        try:
            profile = instaloader.Profile.from_username(L.context, username)
            bio     = (profile.biography or "").strip()
            ig_cache[username] = f"{username} {bio}" if bio else username
            status = f"✓ bio ({len(bio)} chars)" if bio else "✓ (pas de bio)"

        except instaloader.exceptions.ProfileNotExistsException:
            ig_cache[username] = username
            status = "⚠️  profil introuvable"
        except instaloader.exceptions.LoginRequiredException:
            ig_cache[username] = username
            status = "🔒 compte privé"
        except instaloader.exceptions.ConnectionException as e:
            ig_cache[username] = username
            status = f"⚠️  connexion : {e}"
        except Exception as e:
            ig_cache[username] = username
            status = f"⚠️  {type(e).__name__}"

        print(f"  [{i+1:3d}/{len(to_fetch)}] @{username:<30} {status}")

        if (i + 1) % 10 == 0:
            save_cache(ig_cache, INSTAGRAM_CACHE_PATH)

        if i < len(to_fetch) - 1:
            time.sleep(DELAY)

    save_cache(ig_cache, INSTAGRAM_CACHE_PATH)
    print("\n✅ Fetch Instagram terminé")

elif not IL_AVAILABLE:
    print("⚠️  instaloader requis ou session manquante — voir cellule précédente")
else:
    print("✓ Tout déjà en cache — rien à fetcher")

✓ Connecté en tant que @arnvudl
Fetching 17 biographies...

  [  1/17] @usthemob                       ✓ bio (9 chars)
  [  2/17] @flo_climatrek                  ✓ bio (134 chars)
  [  3/17] @eklavya_frr                    ✓ bio (39 chars)
  [  4/17] @sametgorgozfilms               ✓ bio (96 chars)
  [  5/17] @edelynemia                     ✓ bio (125 chars)
  [  6/17] @ibizastardustradio             ✓ bio (131 chars)
  [  7/17] @thehybriddesigner.np           ✓ bio (131 chars)
  [  8/17] @brillyondabeat                 ✓ bio (26 chars)
  [  9/17] @kellybadakdesign               ✓ bio (110 chars)
  [ 10/17] @djmc_gaz974                    ✓ bio (82 chars)
  Cache sauvegardé : /opt/spark/data/warehouse/instagram_bios.json  (10 entrées)
  [ 11/17] @alyxxcould                     ✓ bio (106 chars)
  [ 12/17] @livingthedream.wa              ✓ bio (150 chars)
  [ 13/17] @fitwcurly                      ✓ bio (88 chars)
  [ 14/17] @madameb0nplan                  ✓ bio (127 chars)
  [ 15/17] @

In [7]:
# ── R\u00e9capitulatif final ────────────────────────────────────────────────────────
tiktok_ok = sum(1 for v in tiktok_cache.values() if v)
tiktok_ko = len(tiktok_cache) - tiktok_ok
ig_total  = len(ig_cache)
ig_with_bio = sum(1 for v in ig_cache.values() if v and len(v.split()) > 1)

print("=" * 52)
print("  ENRICHISSEMENT TERMIN\u00c9")
print("=" * 52)
print(f"  TikTok descriptions :")
print(f"    \u2192 avec texte     : {tiktok_ok:,}")
print(f"    \u2192 sans texte     : {tiktok_ko:,}  (fallback comportemental)")
print(f"  Instagram bios :")
print(f"    \u2192 comptes totaux : {ig_total:,}")
print(f"    \u2192 avec bio       : {ig_with_bio:,}")
print()
print(f"  Caches :")
print(f"    {TIKTOK_CACHE_PATH}")
print(f"    {INSTAGRAM_CACHE_PATH}")
print()
print("  \u2192 Lancer 02_graph.ipynb")

  ENRICHISSEMENT TERMINÉ
  TikTok descriptions :
    → avec texte     : 0
    → sans texte     : 6,065  (fallback comportemental)
  Instagram bios :
    → comptes totaux : 17
    → avec bio       : 17

  Caches :
    /opt/spark/data/warehouse/tiktok_descriptions.json
    /opt/spark/data/warehouse/instagram_bios.json

  → Lancer 02_graph.ipynb
